# sj-cohort-update-2022
Analyses determining the number, composition, and file IDs of tumor WGS added to St Jude cloud 2019-2022.  
Requires authorization to access Chavez lab projects on DNANexus.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import upsetplot
from pathlib import Path

pd.set_option('display.max_columns', None)

In [ ]:
PROJECT_ROOT='../..'
SAMPLE_INFO_DIR='data/source/sjcloud/sample_info'

In [ ]:
si_ppc_ec=Path(PROJECT_ROOT,SAMPLE_INFO_DIR,"SAMPLE_INFO PedPanCancer ecDNA.txt").resolve()
si_ppc_cl=Path(PROJECT_ROOT,SAMPLE_INFO_DIR,"SAMPLE_INFO PedPanCan_clinical.txt").resolve()

def read_sampleinfo(file):
    df = pd.read_csv(file,sep='\t')
    df=df[(df.sequencing_type=='WGS') & (df.file_type=='BAM') & (df.sample_type.map(lambda x:x.lower())!='germline') & 
          df.file_path.map(lambda x: x.endswith(".bam"))].copy()
    df=df[~(df["attr_oncotree_disease_code"].str.contains('|'.join(['ALL','AML','BLL','TLL','AMKL','LCH','MPRDS','ALCL','APL','CML','MDS'])))]
    df=df[(df['sample_type']!='xenograft')]
    return df
ec = read_sampleinfo(si_ppc_ec)
ec["source"]="ec"
#cl = read_sampleinfo(si_ppc_cl)
#cl["source"]="cl"

si_ppc_la=Path(PROJECT_ROOT,SAMPLE_INFO_DIR,"SAMPLE_INFO_2022-03-02.tsv").resolve()
#si_ppc_la=Path(PROJECT_ROOT,SAMPLE_INFO_DIR,"SAMPLE_INFO_2023-06-22.tsv").resolve()
la = read_sampleinfo(si_ppc_la)
la["source"]="la"

In [ ]:
#ec=ec[~(ec["attr_oncotree_disease_code"].str.contains('|'.join(['ALL','AML','BLL','TLL','AMKL','LCH','MPRDS','ALCL','APL','CML','MDS'])))]
la

In [ ]:
#patient set overlap
def patient_set_overlap():
    a = set(ec.subject_name)
    #b = set(cl.subject_name)
    c = set(la.subject_name)
    dataset = upsetplot.from_contents({
        "current":a,
        #"additional":b,
        "latest":c
    })
    #dataset = upsetplot.UpSet(dataset,subset_size='count')
    upsetplot.plot(dataset, show_counts=True)
    return
patient_set_overlap()
# 725 patients currently. 
# 42 in additional that arent in latest
# 681 additional patients in latest.

In [ ]:
#sample set overlap
def sample_set_overlap():
    a = set(ec.sample_name)
    #b = set(cl.sample_name)
    c = set(la.sample_name)
    dataset = upsetplot.from_contents({
        "current":a,
        #"additional":b,
        "latest":c
    })
    upsetplot.plot(dataset, show_counts=True)
    return
sample_set_overlap()
# 807 samples currently. 
# None in additional that arent in latest
# 1004 additional samples in latest (2022-03-02)
# 1073 additional samples in latest (2023-06-22)

In [ ]:
def create_joint_cohort():
    df = pd.concat([ec,la],ignore_index=True)
    df = df.sort_values(by=["sample_name","source"])
    # Latest has more complete metadata than our current.
    df = df.drop_duplicates(subset=["sample_name"],keep='last') # keep the la metadata
    df.loc[df.sample_name.isin(ec.sample_name),"source"]="ec" # but note that it's in our current cohort.
    return df
joint_cohort = create_joint_cohort()
new_cohort = joint_cohort[joint_cohort.source=='la']
current_cohort = joint_cohort[joint_cohort.source=='ec']

In [ ]:
print(f"number of unique biosamples: {len(joint_cohort.sample_name.unique())}")
print(f"number of unique patients: {len(joint_cohort.subject_name.unique())}")

In [ ]:
# What kinds of new samples did we get?
def sj_sample_types():
    #print("New cohort by sample type:")
    new_ct = new_cohort.groupby('sample_type').count().sample_name
    new_ct.name = 'latest'
    #print("\nCurrent cohort by sample type:")
    cur_ct = current_cohort.groupby('sample_type').count().sample_name
    cur_ct.name = 'current'
    df = pd.concat([cur_ct,new_ct],axis=1).fillna(0)
    df["Total"] = df.current + df.latest
    df.sort_values("Total",inplace=True,ascending=False)
    # plot
    sns.despine()
    df[["current","latest"]].plot(kind='bar',stacked=True)
    plt.gcf().set_size_inches(6, 4)
    return df
sj_sample_types()

In [ ]:
# what tumor types did we get?
def sj_sample_tumor_types():
    #print("New cohort by sample type:")
    new_ct = new_cohort.groupby('attr_oncotree_disease_code').count().sample_name
    new_ct.name = 'latest'
    #print("\nCurrent cohort by sample type:")
    cur_ct = current_cohort.groupby('attr_oncotree_disease_code').count().sample_name
    cur_ct.name = 'current'
    df = pd.concat([cur_ct,new_ct],axis=1).fillna(0)
    df["Total"] = df.current + df.latest
    df.sort_values("Total",inplace=True,ascending=False)
    # drop codes with less than a certain number of counts
    df = df[df.Total > 2]
    # plot
    sns.despine()
    df[["current","latest"]].plot(kind='bar',stacked=True)
    plt.gcf().set_size_inches(15, 6)
    return df

sj_sample_tumor_types()

In [ ]:
print(f"number of unique biosamples: {len(la.sample_name.unique())}")
print(f"number of unique patients: {len(la.subject_name.unique())}")

In [ ]:
# what xenografts did we get?
def sj_xenograft_tumor_types():
    df = new_cohort[new_cohort.sample_type == 'Xenograft']
    df = df.groupby('attr_oncotree_disease_code').count().sample_name.copy()
    df.name = 'latest'
    df.sort_values(inplace=True,ascending=False)
    # plot
    sns.despine()
    df.plot(kind='bar')
    plt.gcf().set_size_inches(8, 4)
    return df

sj_xenograft_tumor_types()

In [ ]:
# What other data do I have for the xenograft tumors?
def sj_xenograft_patient_samples():
    xenograft_patients = set(new_cohort[new_cohort.sample_type == 'Xenograft'].subject_name)
    df = joint_cohort[joint_cohort.subject_name.isin(xenograft_patients)]
    # How many have paired ht data? (Diagnosis, Metastasis, or Relapse)
    print("Xenografts with paired human tumor:")
    subjects_w_human_tumor=df[df.sample_type.isin(["Diagnosis","Metastasis","Relapse"])].subject_name.unique()
    print("{}/{}".format(
        len(df[(df.sample_type=="Xenograft") & (df.subject_name.isin(subjects_w_human_tumor))]),
        len(df[df.sample_type=="Xenograft"]))
    )
    df = df.groupby(["subject_name","sample_type"]).count()
    return df
sj_xenograft_patient_samples()
#joint_cohort[(joint_cohort.subject_name=="SJ000026")]
#joint_cohort[(joint_cohort.subject_name=="SJ000911")]

In [ ]:
# What files does Shanqing need?
def read_germlineinfo(file):
    df = pd.read_csv(file,sep='\t')
    df=df[(df.sequencing_type=='WGS') & (df.file_type=='BAM') & (df.sample_type=='Germline') & 
          df.file_path.map(lambda x: x.endswith(".bam"))].copy()
    return df

def create_latest_file_list():
    file=Path(PROJECT_ROOT,SAMPLE_INFO_DIR,"SAMPLE_INFO_2022-03-02.tsv").resolve()
    df = read_germlineinfo(file)
    
    patients = new_cohort.subject_name
    df=df[df.subject_name.isin(patients)]
    print("{} patients with germline sequencing.".format(len(df.drop_duplicates("subject_name"))))
    
    df = pd.concat([df,new_cohort])
    
    # Add .bai files
    bais = df.file_path.map(lambda x: x+".bai").to_list()
    bai_files = pd.read_csv(file,sep='\t')
    bai_files = bai_files[bai_files.file_path.isin(bais)]
    print(len(df))
    print(len(bai_files))
    
    df = pd.concat([df,bai_files]).sort_values("sample_name")
    
    return df
sampleinfo = create_latest_file_list()
sampleinfo.to_csv("out/SAMPLE_INFO_batch_2022-03-03.txt",sep='\t',index=False)
# Why are there multiple germlines for some patients? 
# idk, let's give Shanqing something to think about.